In [ ]:
%%sql
select * from edw.tbl_fact_policy_quotes limit 100

In [ ]:
%%sql
select * from edw.tbl_fact_policy_sales limit 100

In [ ]:
%%sql
select * from edw.tbl_fact_policy_renewals limit 100

In [ ]:
%%sql
select * from dlk.hfq_quotedetails limit 100

In [ ]:
%%sql
select * from dlk.genesys_session_summary 
Where conversationStartTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' and direction = 'outbound' limit 100

Create Event Stream

In [ ]:
Create Table if not exists ods.EpisodeEventStream 
(
    EventId                 BigInt IDENTITY,
    EventSourceId           Int,
    EventSourceReference    Varchar(128),
    SourceQuoteReference    Varchar(128),
    SourcePolicyReference   Varchar(128),
    SourceCustomerReference Varchar(128),
    SourceSystemId          int,
    PolicyTypeGroup         varchar(50),
    EventDateTime           TIMESTAMP,
    EventDate               Date,
    EventTypeId             int,
    EventDescription        varchar(1024),
    Grain                   varchar(50)
)

## Populate step R1

In [1]:
/* Renewals and sales to be derived properly - we are picking up from DWH for now */
/* Renewals are always derived for the month of renewals -- and their sales over time across months */ 
Create or Replace Table stg.R0_HomePoliciesEligibleForRenewals as 
select  PolicyCode, LyPolicyRenewDateAdj as RenewalDate, LYReportingSaleDate, TyPolicyCode , TYReportingSaleDate, TyPolicyRenewDateAdj, Channel, PolicyOfferNum, PolicyRetNum,
 RenEURPremOffer, TYEURGrossPremium
from    edw.tbl_fact_policy_renewals    a 
Where   a. LyPolicyTypeGroup = 'Home' 
and     a.LYPolicyRenewDateAdj between '2026-07-01' and '2026-07-31' 
-- and     PolicyOfferNum = 1 
Group by   PolicyCode, LyPolicyRenewDateAdj, LYReportingSaleDate,  TyPolicyCode  , TYReportingSaleDate,  TyPolicyRenewDateAdj, Channel, PolicyOfferNum, PolicyRetNum,
 RenEURPremOffer, TYEURGrossPremium

StatementMeta(, 797d024a-6585-455d-ade0-6cf3b4fcdefd, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
/*
Home	Clams HCL4
*/
select  count(distinct a.PolicyCode)
from    edw.tbl_fact_policy_mvt                 a,
        (select	distinct b.PolicyCode
        from	dlk.EXT_Home_QS_ClaimDetails    a, 
                dlk.ext_home_qs_policydetails   b
        Where	a.HomeRiskId = upper(b.RiskId)
        and		TransactionDate between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000') b 
Where   a.PolicyCode = b.PolicyCode         
and     a.EffectiveDate in ('2026-07-31', '2026-08-31') 
and     a.CloseNum = 1 


StatementMeta(, 67409023-4ae8-4f79-ba54-8c9c78750947, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [2]:
select count(*) from stg.R0_HomePoliciesEligibleForRenewals

StatementMeta(, 797d024a-6585-455d-ade0-6cf3b4fcdefd, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [3]:
/* HR1a */ 
Create or Replace Table stg.HR1a_Home_Renewals_EmailOffered as 
select  SourcePolicyReference, min(EventDateTime) OfferedDateTime
from    ods.EventStream                             a, 
        stg.R0_HomePoliciesEligibleForRenewals     b 
Where   EventDescription like 'Renewal Offer - Emailed Document %'
and     a.PolicyTypeGroup = 'Home'
and     a.SourcePolicyReference = b.PolicyCode 
and     EventDateTime between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
Group by SourcePolicyReference
;

StatementMeta(, 03a51d74-3c65-4b90-a9bc-4ddcabb25e34, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
/* HR1.E1
*/
  -- get Renewal emails from XP table
select  count(distinct user_id), count(*)
from    dlk.EXT_XtremePushResults_Policy a, stg.R0_HomePoliciesEligibleForRenewals b 
where campaign_name like '%Home - Renewals%' and a.PolicyCode = b.PolicyCode 
and     `timestamp` between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     interaction_type = 'sent' 
and     MessageType  in ( 'EMAIL', 'SMS') 


StatementMeta(, 03a51d74-3c65-4b90-a9bc-4ddcabb25e34, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [4]:
/* HR0.F2 */ 
/*Create or Replace Table stg.HR1a_Home_Renewals_Held as 
select  SourcePolicyReference, min(EventDateTime) OfferedDateTime*/
select count(distinct SourcePolicyReference)
from    ods.EventStream                             a, 
        stg.R0_HomePoliciesEligibleForRenewals     b 
Where   EventDescription like 'Renewal Offer - Emailed Document %'
and     upper(EventDescription) like '% HELD %'
and     a.PolicyTypeGroup = 'Home'
and     a.SourcePolicyReference = b.PolicyCode 
and     EventDateTime between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
/*Group by SourcePolicyReference*/
;

StatementMeta(, 03a51d74-3c65-4b90-a9bc-4ddcabb25e34, 5, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [4]:
/* HR1b */ 
Create or Replace Table stg.HR1b_Home_Renewals_PostOffered as  
select  SourcePolicyReference, min(EventDateTime) OfferedDateTime
from    ods.EventStream a, stg.R0_HomePoliciesEligibleForRenewals b 
Where   EventDescription like 'Renewal Offer - Document Transmitted %'
and     a.PolicyTypeGroup = 'Home'
and     a.SourcePolicyReference = b.PolicyCode 
and     EventDateTime between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
Group by SourcePolicyReference
;

StatementMeta(, 797d024a-6585-455d-ade0-6cf3b4fcdefd, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
/* R1a */
select count(*) from stg.HR1a_Home_Renewals_EmailOffered ;
select count(*) from stg.HR1b_Home_Renewals_PostOffered ;
select count(*) from stg.HR1a_Home_Renewals_EmailOffered a, stg.HR1b_Home_Renewals_PostOffered b where a.SourcePolicyReference = b.SourcePolicyReference;



StatementMeta(, 797d024a-6585-455d-ade0-6cf3b4fcdefd, 8, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

In [6]:
/* HR2 */ 
select  count(*), count(distinct ClientCode), count(distinct PolicyCode)
from    (
            select  left(PolicyCode,6) as ClientCode, dateadd(day, -40, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
            from stg.R0_HomePoliciesEligibleForRenewals 
        )   a,
        dlk.appliedrenewals_successfulloginevent b 
Where   a.ClientCode = b.PortfolioCode 
and     b.`Timestamp` > a.RenewalStartDate
and     b.`Timestamp` < a.RenewalEndDate
;

StatementMeta(, 797d024a-6585-455d-ade0-6cf3b4fcdefd, 9, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

In [11]:
/* NOT HR2.F1 */ 
select  count(*), count(distinct ClientCode), count(distinct PolicyCode)
from    (
            select  left(PolicyCode,6) as ClientCode, dateadd(day, -40, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
            from stg.R0_HomePoliciesEligibleForRenewals 
        )   a,
        dlk.appliedrenewals_failedloginevent b 
Where   a.ClientCode = b.PortfolioCode 
and     b.`Timestamp` > a.RenewalStartDate
and     b.`Timestamp` < a.RenewalEndDate;

StatementMeta(, 797d024a-6585-455d-ade0-6cf3b4fcdefd, 14, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

In [9]:
/* HR2.B1 */
-- logic is a bit convoluted 
-- This could be a) all people who do an HFQ who are eligible for renewals or b) renewal people who pay via HFQ c) failed login people who go to hfq 

-- Let us do a for now
Create or Replace Table stg.HomeRenewalScvCustomerKeys as 
select  distinct scv_customer_key , PolicyCode, ClientCode 
from    ods.scv_customer_key  a, (select PolicyCode, left(PolicyCode,6) as ClientCode from stg.R0_HomePoliciesEligibleForRenewals) b 
Where   a.SourceSystemReference = b.ClientCode 
and     a.SourceSystemId = 1 
;

StatementMeta(, 797d024a-6585-455d-ade0-6cf3b4fcdefd, 12, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [17]:
Create Or Replace Table stg.HFQ_Quotes as 
SELECT  *
FROM    (SELECT  *, 	
                CAST(from_unixtime(_ts) AS TIMESTAMP) as ts_unix,
                ROW_NUMBER() OVER(PARTITION BY QuoteCodeReference, RetrieveCount order by _ts desc) AS RN 
        FROM    dlk.HFQ_QuoteDetails_Snapshot_v2) a
where   RN =1
and     ts_unix between '2026-06-01 00:00:00.000' and '2026-08-01 00:00:00.000'  

StatementMeta(, 797d024a-6585-455d-ade0-6cf3b4fcdefd, 20, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [1]:
select * from stg.HFQ_Quotes limit 100 

StatementMeta(, 926dd6a4-41b8-43b8-9b88-e13f91fdb296, 2, Finished, Available, Finished, False)

<Spark SQL result set with 100 rows and 110 fields>

In [2]:
/* HR2.B1 */
-- logic is a bit convoluted 
-- This could be a) all people who do an HFQ who are eligible for renewals or b) renewal people who pay via HFQ c) failed login people who go to hfq 

-- Let us do a for now 
Create or Replace Table stg.HomeRenewalsDoingHFQ as 
select  distinct a.scv_customer_key, a.PolicyCode, b.QuoteCodeReference, ts_unix as QuoteStartDateTime
from    stg.HomeRenewalScvCustomerKeys a, stg.HFQ_Quotes b, ods.scv_customer_key c 
Where   a.scv_customer_key = c.scv_customer_key 
and     b.QuoteCodeReference = c.SourceSystemReference 
and     ts_unix between '2026-05-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     SourceSystemId = 3
;

StatementMeta(, 926dd6a4-41b8-43b8-9b88-e13f91fdb296, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [3]:

/* HR2.B1 */
select count(distinct PolicyCode), count(distinct QuoteCodeReference)
from stg.HomeRenewalsDoingHFQ 
-- if they paid via HFQ is derived below 

StatementMeta(, 926dd6a4-41b8-43b8-9b88-e13f91fdb296, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [1]:

select count(distinct a.PolicyCode), count(distinct a.QuoteCodeReference)
from    stg.HomeRenewalsDoingHFQ    a 
left join 
        (select QuoteCodeReference  
        from    dlk.HFQ_Response_Quotes  
        where   Quotes_Premium <> 0 
        and     Quotes_Outcome = 'PremiumReturned'        
        Group by QuoteCodeReference) b
        on a.QuoteCodeReference = b.QuoteCodeReference 
Where   b.QuoteCodeReference is null 

StatementMeta(, d014873f-b1b0-4ea5-bbad-d08058ae0fc0, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [ ]:
b

In [30]:
/* HFQ payment ODS event joins. Read-only; mirrors DWH.usp_Load_fact_HFQ_Payment. */
Create or Replace Table stg.HFQ_Payments as 
WITH Request_Request AS
(
    SELECT *
    FROM
    (
        SELECT trim(upper(globalPayment_reference)) AS PaymentReference, trim(upper(reference)) AS QuoteReference,
               SchemeCode, TemporaryQuoteId, CacheId, amount, depositAmount,
               isFullPayment, monthlyAmount, payer_email, policyStartDate, BrokerID,
               ROW_NUMBER() OVER
                 (PARTITION BY globalPayment_reference ORDER BY globalPayment_reference DESC) AS rn
        FROM dlk.HFQ_PaymentRequest_Request
        WHERE SourceFilePath NOT LIKE '%Global%'
          AND TemporaryQuoteId IS NOT NULL AND CacheId IS NOT NULL
          AND globalPayment_reference IS NOT NULL AND reference IS NOT NULL
    ) AS source_data
    WHERE rn = 1
),
Request_Response AS
(
    SELECT *
    FROM
    (
        SELECT trim(upper(REPLACE(SUBSTRING(SourceFilePath, 26, LEN(SourceFilePath)), '.json', ''))) AS PaymentReference,
               action_time_created, action_result_code, action_type,
               ROW_NUMBER() OVER
                 (PARTITION BY REPLACE(SUBSTRING(SourceFilePath, 26, LEN(SourceFilePath)), '.json', '')
                  ORDER BY action_time_created DESC) AS rn
        FROM dlk.HFQ_PaymentRequest_Response
    ) AS source_data
    WHERE rn = 1
),
Response_Response AS
(
    SELECT *
    FROM
    (
        SELECT trim(upper(reference)) AS PaymentReference, action_result_code, action_time_created,
               type, status, amount, payment_method_result, payment_method_message,
               ROW_NUMBER() OVER (PARTITION BY reference ORDER BY action_time_created DESC) AS rn
        FROM dlk.HFQ_PaymentResponse_Response
    ) AS source_data
    WHERE rn = 1
),
Request_Failed AS
(
    SELECT *
    FROM
    (
        SELECT trim(upper(REPLACE(SUBSTRING(SourceFilePath, 24, LEN(SourceFilePath)), '.json', ''))) AS PaymentReference,
               StatusCode, ReasonPhrase, IsSuccessStatusCode,
               REPLACE(REPLACE(Headers_Value, '["', ''), '"]', '') AS FailureTimestamp,
               ROW_NUMBER() OVER
                 (PARTITION BY REPLACE(SUBSTRING(SourceFilePath, 24, LEN(SourceFilePath)), '.json', '')
                  ORDER BY Headers_Value DESC) AS rn
        FROM dlk.HFQ_PaymentRequest_Failed
    ) AS source_data
    WHERE rn = 1
)
SELECT
    req.PaymentReference AS RequestPaymentReference,
    req.QuoteReference, req.BrokerID, req.SchemeCode, req.TemporaryQuoteId, req.CacheId,
    req.amount AS RequestAmount, req.depositAmount, req.isFullPayment, req.monthlyAmount,
    req.payer_email, req.policyStartDate,
    reqRes.PaymentReference AS RequestResponsePaymentReference,
    reqRes.action_time_created AS RequestResponseTimestamp,
    reqRes.action_result_code AS RequestResponseActionResult,
    reqRes.action_type AS RequestResponseActionType,
    resRes.PaymentReference AS PaymentResponsePaymentReference,
    resRes.action_result_code AS PaymentResponseActionResultCode,
    resRes.action_time_created AS PaymentResponseTimestamp,
    resRes.type AS PaymentResponseType, resRes.status AS PaymentResponseStatus,
    resRes.amount AS PaymentResponseAmount,
    resRes.payment_method_result AS PaymentMethodResult,
    resRes.payment_method_message AS PaymentMethodMessage,
    fail.PaymentReference AS FailedRequestPaymentReference,
    fail.StatusCode AS FailedRequestStatusCode, fail.ReasonPhrase AS FailedRequestReason,
    fail.IsSuccessStatusCode AS FailedRequestIsSuccessStatusCode, fail.FailureTimestamp,

    /* Fact flags and dates, named to match DWH.tbl_fact_HFQ_Payment. */
    CASE WHEN fail.PaymentReference IS NULL THEN 0 ELSE 1 END AS PaymentRequestFailed,
    fail.FailureTimestamp AS PaymentRequestFailedTimestamp,
    CASE WHEN reqRes.PaymentReference IS NULL THEN 0 ELSE 1 END AS PaymentRequestResponse,
    reqRes.action_time_created AS PaymentRequestResponseTimestamp,
    CASE WHEN resRes.PaymentReference IS NULL THEN 0 ELSE 1 END AS PaymentResponseResponse,
    resRes.action_result_code AS PaymentResponseResponseActionResultCode,
    resRes.action_time_created AS PaymentResponseResponseTimestamp,
    CASE WHEN resRes.action_result_code IS NOT NULL THEN 1 END AS RecordVersionIsLatestFlag,
    CASE
        WHEN resRes.action_time_created IS NOT NULL THEN CAST(resRes.action_time_created AS date)
        WHEN reqRes.action_time_created IS NOT NULL THEN CAST(reqRes.action_time_created AS date)
        WHEN fail.FailureTimestamp IS NOT NULL THEN 
                            cast(concat(
                                SUBSTRING(fail.FailureTimestamp, 13, 4) , '-' ,
                                CASE SUBSTRING(fail.FailureTimestamp, 9, 3)
                                    WHEN 'Jan' THEN '01' WHEN 'Feb' THEN '02' WHEN 'Mar' THEN '03'
                                    WHEN 'Apr' THEN '04' WHEN 'May' THEN '05' WHEN 'Jun' THEN '06'
                                    WHEN 'Jul' THEN '07' WHEN 'Aug' THEN '08' WHEN 'Sep' THEN '09'
                                    WHEN 'Oct' THEN '10' WHEN 'Nov' THEN '11' WHEN 'Dec' THEN '12'
                                        END , '-' , SUBSTRING(fail.FailureTimestamp, 6, 2), ' ',
                                        Substring(fail.FailureTimestamp,18,8)) 
                                        as timestamp)
    END AS WorkDate
FROM Request_Request AS req
FULL OUTER JOIN Request_Response AS reqRes
    ON req.PaymentReference = reqRes.PaymentReference
FULL OUTER JOIN Response_Response AS resRes
    ON req.PaymentReference = resRes.PaymentReference
FULL OUTER JOIN Request_Failed AS fail
    ON req.PaymentReference = fail.PaymentReference;


StatementMeta(, 2deb1656-5e81-4fe0-9419-94284cd97aff, 31, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [40]:
/* HR2.B1.F1 */
select count(distinct a.PolicyCode), count(distinct a.QuoteCodeReference)
from stg.HomeRenewalsDoingHFQ  a, stg.HFQ_Payments  b 
Where   PaymentResponseResponseActionResultCode = 'SUCCESS' and b.QuoteReference = a.QuoteCodeReference
and     PaymentResponseResponseTimestamp between '2026-07-01' and '2026-07-31' 

StatementMeta(, 2deb1656-5e81-4fe0-9419-94284cd97aff, 41, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [38]:
/* HA4a */
select count(distinct QuoteReference) 
from    stg.HFQ_Payments 
Where   PaymentResponseResponseActionResultCode = 'SUCCESS' and QuoteReference is not null
and     PaymentResponseResponseTimestamp between '2026-07-01' and '2026-07-31' 
limit 10 


StatementMeta(, 2deb1656-5e81-4fe0-9419-94284cd97aff, 39, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [1]:
select  Channel, count(distinct TyPolicyCode)
from    edw.tbl_fact_policy_renewals b 
where   LYPolicyTypeGroup = 'Home' 
and     TYReportingSaleType = 'Renewals' 
and     RenewalMonth = '2026-07-31' 
and     PolicyRetNum = 1 
Group by Channel


StatementMeta(, edbe5654-aed0-4f10-95ca-89d68e074a0e, 2, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

In [50]:
/* R2.B1.F1 example -- the quoted prices are cheaper but the client went for a higher price .. why ? */
select * from edw.tbl_fact_policy_renewals where PolicyCode = 'HEGJ91004' and RenewalMonth = '2026-07-31'
-- HEGJ91004,HEGJ91,1ebb6584-559e-4486-91de-a7a80d463234

StatementMeta(, ecbbf383-6e53-431e-bbc2-85fa80649232, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 314 fields>

In [4]:
-- HR3a - component 1
-- (HR3a  / b might include a transfer in which case the sold policy this year may have a different policy code - one to check for)
select  count(*), count(distinct ClientCode), count(distinct PolicyCode)
from    (
            select  left(PolicyCode,6) as ClientCode, dateadd(day, -60, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
            from stg.R0_HomePoliciesEligibleForRenewals 
        )   a,
        dlk.AppliedRenewals_PaymentSuccess b 
Where   PolicyCode = PolicyCodeForRenewal
and     b.`Timestamp` > a.RenewalStartDate
and     b.`Timestamp` < a.RenewalEndDate;

StatementMeta(, 926dd6a4-41b8-43b8-9b88-e13f91fdb296, 5, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

In [7]:
-- R3a - component 2 (Ewan --> add this and the prev cell for HR3a)
-- (R3a  / b might include a transfer in which case the sold policy this year may ahve a different policy code - one to check for)
select  count(*), count(distinct ClientCode), count(distinct PolicyCodeForRenewal)
from    (
            select  left(PolicyCode,6) as ClientCode, dateadd(day, -60, RenewalDate) RenewalStartDate, PolicyCode, TyPolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
            from stg.R0_HomePoliciesEligibleForRenewals 
        )   a,
        dlk.AppliedRenewals_PaymentSuccess b 
Where   ClientCode = left(PolicyCodeForRenewal,6)
and     PolicyCode <> PolicyCodeForRenewal
and     TyPolicyCode = PolicyCodeForRenewal
and     b.`Timestamp` > a.RenewalStartDate
and     b.`Timestamp` < a.RenewalEndDate; 

StatementMeta(, 926dd6a4-41b8-43b8-9b88-e13f91fdb296, 8, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

In [5]:
/* HR3b */
 
select  distinct TyPolicyCode,  TYReportingSaleDate
from    stg.R0_HomePoliciesEligibleForRenewals
Where   PolicyRetNum = 1 
and     Channel = 'a) IB' 

StatementMeta(, edbe5654-aed0-4f10-95ca-89d68e074a0e, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [128]:
/* R3b.F1 Genesys Call Wait Times */ 
--  stg.A0_genesys_derived_data is already derived but remember to not use one stg from one notebook in another notebook --- that can lead to disaster.. 
-- each notebook is self contained for stg tables.. ods / dlk / prd are the dependable tables across schemas  

Create or Replace Table stg.R0_genesys_derived_data as 
select  conversationId, 
        case 
            when originatingDirection = 'inbound' then replace(ani,'tel:+353', '0') 
            when originatingDirection = 'outbound' then replace(dnis,'tel:+353', '0') 
            else null 
        end as CustomerPhoneNumber,
        originatingDirection,          
        conversationStartTime,
        conversationEndTime,
        sessionCount,
        sessionIndex,
        sessionDuration,
        sessionStartTime,
        sessionEndTime,
        queueName,
        purpose,
        transferToId,
        agentAnswered, 
        alertNoAnswer, 
        abandoned, 
        totalAcdWaitDuration, 
        totalAgentAlertDuration, 
        totalAgentHoldDuration, 
        totalAgentTalkDuration
from    dlk.genesys_session_summary
Where	conversationStartTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
;

StatementMeta(, 1870d5fd-1b37-43ca-8415-86357575d543, 74, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [1]:
Create or Replace Table stg.HR3bF1_genesys_inbound_call_duration_summary as 
select	a.ConversationId, min(conversationStartTime) as conversationStartTime, 
            sum(agentAnswered) as agentAnswered, sum(alertNoAnswer) alertNoAnswer, sum(abandoned) abandoned, 
            sum(totalAcdWaitDuration) CallWaitTime, sum(totalAgentAlertDuration) CallRingTime, sum(totalAgentHoldDuration) CallHoldTime, sum(totalAgentTalkDuration) CallSpokenTime
from	stg.R0_genesys_derived_data	a
where	a.queueName = 'INBOUND_Home_Renewals'  
Group by ConversationId

StatementMeta(, a05c8b46-aad2-4167-8ddd-4f66da6b9784, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
/* R3b.F1 */ 
select count(distinct b.CustomerPhoneNumber) from stg.HR3bF1_genesys_inbound_call_duration_summary a, stg.R0_genesys_derived_data b where a.abandoned = 1 and a.CallWaitTime > 0 and a.ConversationId = b.ConversationId 

StatementMeta(, a05c8b46-aad2-4167-8ddd-4f66da6b9784, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [4]:
/* HR3.B1 */ 
Create or Replace Table stg.HR3B1_LapsedThisYearPolicy as 
select  distinct b.PolicyCode , b.RenewalDate
from    dlk.rbsdata2_policy_physical a, stg.R0_HomePoliciesEligibleForRenewals b
Where   trim(a.pl_code) = trim(b.PolicyCode)  -- trim is important as the source tables have trailing spaces -- Urska needs to address this 
and     trim(a.pl_status) = 'L' 


StatementMeta(, a05c8b46-aad2-4167-8ddd-4f66da6b9784, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
/* HR3.B1 HR3.O1 */ 
select  count(*) 
from    stg.HR3B1_LapsedThisYearPolicy 


StatementMeta(, a05c8b46-aad2-4167-8ddd-4f66da6b9784, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [13]:
-- HR4
select count(distinct TyPolicyCode)
from stg.R0_HomePoliciesEligibleForRenewals 
Where PolicyRetNum = 1 

StatementMeta(, a05c8b46-aad2-4167-8ddd-4f66da6b9784, 14, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [6]:
-- not assigned to any episode but this is friction

select  count(*), count(distinct ClientCode), count(distinct PolicyCode)
from    (
            select  left(PolicyCode,6) as ClientCode, dateadd(day, -60, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
            from stg.R0_HomePoliciesEligibleForRenewals 
        )   a,
        dlk.AppliedRenewals_PaymentFailedEvent b 
Where   ClientCode = b.PortfolioCode
and     b.`Timestamp` > a.RenewalStartDate
and     b.`Timestamp` < a.RenewalEndDate;


StatementMeta(, a05c8b46-aad2-4167-8ddd-4f66da6b9784, 7, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

In [24]:
/* R5 */
SELECT  EventDescription, count(*)
FROM    stg.R0_HomePoliciesEligibleForRenewals  a,
        ods.EventStream                         d 
Where   a.TyPolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Home'
and     PolicyRetNum = 1 
and     EventDateTime between '2026-06-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     EventDescription not like 'Renewal Offer - %'
AND     (
            EventDescription like '%Emailed Document%'
        OR    
            EventDescription like '%Document Transmitted%'
        )
Group by EventDescription
Order by 2 desc 

StatementMeta(, a05c8b46-aad2-4167-8ddd-4f66da6b9784, 25, Finished, Available, Finished, False)

<Spark SQL result set with 104 rows and 2 fields>

In [15]:
-- HR4a 
Create or Replace Table stg.HR4_HomeRenewalsOnlinePayments as 
select  ClientCode, PolicyCode, max(`Timestamp`) PaymentDateTime 
from    (
            select  left(PolicyCode,6) as ClientCode, dateadd(day, -60, RenewalDate) RenewalStartDate, PolicyCode, dateadd(day, 40, RenewalDate) RenewalEndDate
            from stg.R0_HomePoliciesEligibleForRenewals 
        )   a,
        dlk.AppliedRenewals_PaymentSuccess b 
Where   ClientCode = b.PortfolioCode
and     b.`Timestamp` > a.RenewalStartDate
and     b.`Timestamp` < a.RenewalEndDate 
Group by ClientCode, PolicyCode
;        

StatementMeta(, a05c8b46-aad2-4167-8ddd-4f66da6b9784, 16, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [17]:
-- HR4a and HR4c -- Possible loan but old finance or new loan?  

select  PaymentType, count(distinct PolicyCode)
from    (
        select  PolicyCode, PaymentDateTime, b.PaymentType
        from    stg.HR4_HomeRenewalsOnlinePayments      a,
                dlk.AppliedRenewals_PaymentSuccess      b 
        Where   ClientCode = b.PortfolioCode
        and     b.`Timestamp` = a.PaymentDateTime
        ) x 
Group by PaymentType
;


StatementMeta(, a05c8b46-aad2-4167-8ddd-4f66da6b9784, 18, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

In [34]:
/* HR6 HR6.B1 */
select count(*) from ( 
        SELECT  a.PolicyCode, min(EventDateTime) EventDateTime
        FROM    stg.R0_HomePoliciesEligibleForRenewals  a,
                ods.EventStream                          d 
        Where   a.TyPolicyCode = d.SourcePolicyReference
        and     d.EventSourceId = 3 
        and     d.PolicyTypeGroup = 'Home'
        and     EventDateTime between '2026-06-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
        AND      EventDescription not like 'Renewal Offer - %'            
        AND     (
                    EventDescription like '%Emailed Document%'
                OR    
                    EventDescription like '%Document Transmitted%'
                )    
        Group by a.PolicyCode ) x 

        /*and     EventDescription IN 
            (
                'Renewal Transfer - Emailed Document - RE3 Home Ext Cover email Receipt',
                'Renewal Transfer - Emailed Document - Receipt Batch Email',
                'New Business - Emailed Document - 04 - Suitability Statement',
                'New Business - Emailed Document - Terms Of Business',
                'Renewal Confirmation - Emailed Document - Receipt_Template', 
                'Renewal Confirmation - Emailed Document - 04 - Suitability Statement',
                'Renewal Transfer - Emailed Document - Rnl Conf Cover Email Home Rollover v2', /* Rollover ? */
                'New Business - Emailed Document - Statement Of Fact',
                'New Business - Emailed Document - Schedule',
                'Renewal Confirmation - Emailed Document - Letter - Home Renewal Transfer (roll over) V1',
                'New Business - Emailed Document - Receipt',
                'New Business - Emailed Document - Insurance Product Information Document',

        Renewal Confirmation - Emailed Document - Email - Home Renewal Confirmation (roll over) V1
        Renewal Confirmation - Document Transmitted - Receipt_Template
            )*/


StatementMeta(, a05c8b46-aad2-4167-8ddd-4f66da6b9784, 35, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [33]:
/* HR6.F1 */
select count(distinct PolicyCode) from ( 
        SELECT  a.PolicyCode, EventDescription, min(EventDateTime) EventDateTime
        FROM    stg.R0_HomePoliciesEligibleForRenewals  a,
                ods.EventStream                          d 
        Where   a.TyPolicyCode = d.SourcePolicyReference
        and     d.EventSourceId = 3 
        and     d.PolicyTypeGroup = 'Home'
        and     EventDateTime between '2026-06-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
        and     EventDescription like '%Suitability%'
        and     EventDescription not like 'Renewal Offer - %'
        AND     (
                    EventDescription like '%Emailed Document%'
                OR    
                    EventDescription like '%Document Transmitted%'
                )    
        Group by a.PolicyCode, EventDescription ) x 

StatementMeta(, a05c8b46-aad2-4167-8ddd-4f66da6b9784, 34, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [37]:
/* HR6.F1 */
select count(distinct PolicyCode) from ( 
        SELECT  a.PolicyCode, EventDescription, min(EventDateTime) EventDateTime
        FROM    stg.R0_HomePoliciesEligibleForRenewals  a,
                ods.EventStream                          d 
        Where   a.TyPolicyCode = d.SourcePolicyReference
        and     d.EventSourceId = 3 
        and     d.PolicyTypeGroup = 'Home'
        and     EventDateTime between '2026-06-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
        and     EventDescription like '%Insurance Product Information Document%'
        and     EventDescription not like 'Renewal Offer - %'
        AND     (
                    EventDescription like '%Emailed Document%'
                OR    
                    EventDescription like '%Document Transmitted%'
                )    
        Group by a.PolicyCode, EventDescription ) x 

StatementMeta(, a05c8b46-aad2-4167-8ddd-4f66da6b9784, 38, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [36]:
/* HR4.B2.O1 */
select count(*) from ( 
        SELECT  a.PolicyCode, min(EventDateTime) EventDateTime
        FROM    stg.R0_HomePoliciesEligibleForRenewals  a,
                ods.EventStream                          d 
        Where   a.TyPolicyCode = d.SourcePolicyReference
        and     d.EventSourceId = 3 
        and     d.PolicyTypeGroup = 'Home'
        and     EventDateTime between '2026-06-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
        and     EventDescription not like 'Renewal Offer - %'
        and     EventDescription like '%Insurer Led Cancelation % Reg canx email template%'
        AND     (
                    EventDescription like '%Emailed Document%'
                OR    
                    EventDescription like '%Document Transmitted%'
                )    
        Group by a.PolicyCode ) x 

StatementMeta(, a05c8b46-aad2-4167-8ddd-4f66da6b9784, 37, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [38]:
select * from stg.HomeRenewalsDoingHFQ limit 100 

StatementMeta(, a05c8b46-aad2-4167-8ddd-4f66da6b9784, 39, Finished, Available, Finished, False)

<Spark SQL result set with 100 rows and 4 fields>

In [43]:
select * from dlk.hfq_quotedetails_snapshot_v2

StatementMeta(, a05c8b46-aad2-4167-8ddd-4f66da6b9784, 44, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 108 fields>

In [12]:
Create or Replace Table stg.HR2B1_1_Base as 
SELECT  distinct b.PolicyCode, b.QuoteCodeReference, b.QuoteStartDatetime , RenEURPremOffer, TYEURGrossPremium, PolicyRetNum
FROM    stg.HomeRenewalsDoingHFQ                b,
        stg.R0_HomePoliciesEligibleForRenewals  c 
Where   b.PolicyCode = c.PolicyCode 
and     QuoteStartDateTime > '2026-05-01 00:00:00.000'

StatementMeta(, 186e685d-e26f-49b2-bbac-6be8e4f2b83e, 13, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [9]:
select count(distinct PolicyCode) from stg.HR2B1_1_Base 

StatementMeta(, 186e685d-e26f-49b2-bbac-6be8e4f2b83e, 10, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [2]:
Create or Replace Table stg.hfq_prices as 
select  QuoteCodeReference ,  min(Quotes_TotalPayable) as QuoteBestPrice
from    dlk.HFQ_Response_Quotes 
where   Quotes_Premium <> 0 
and     Quotes_Outcome = 'PremiumReturned'
Group by QuoteCodeReference
    

StatementMeta(, 186e685d-e26f-49b2-bbac-6be8e4f2b83e, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
/* HR2.B1.2
 */
select *  from stg.HR2B1_1_Base a, stg.hfq_prices b Where a.QuoteCodeReference = b.QuoteCodeReference 
limit 100 
-- ideally pull the last record or the least QuoteBestPrice for eventstream

StatementMeta(, 186e685d-e26f-49b2-bbac-6be8e4f2b83e, 5, Finished, Available, Finished, False)

<Spark SQL result set with 100 rows and 7 fields>

In [5]:
/* HR2.B1.2
 */
select count(distinct PolicyCode) from stg.HR2B1_1_Base a, stg.hfq_prices b Where a.QuoteCodeReference = b.QuoteCodeReference 
/*  HR2.B1.2.F1 -- just do a left join and list the policies with no pricing */ 

StatementMeta(, 186e685d-e26f-49b2-bbac-6be8e4f2b83e, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [6]:
/* HR2.B1.3.F1
 */
select count(distinct PolicyCode) from stg.HR2B1_1_Base a, stg.hfq_prices b Where a.QuoteCodeReference = b.QuoteCodeReference and TyEurGrossPremium < QuoteBestPrice 



StatementMeta(, 186e685d-e26f-49b2-bbac-6be8e4f2b83e, 7, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [16]:
/* HR2.B1.O1
 */
select count(distinct PolicyCode) from stg.HR2B1_1_Base a, stg.hfq_prices b 
Where a.QuoteCodeReference = b.QuoteCodeReference and TyEurGrossPremium = RenEurPremOffer and PolicyRetNum = 1 

StatementMeta(, 186e685d-e26f-49b2-bbac-6be8e4f2b83e, 17, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [14]:
/* HR2.B1.O2
 */
select count(distinct PolicyCode) from stg.HR2B1_1_Base a, stg.hfq_prices b 
Where a.QuoteCodeReference = b.QuoteCodeReference and TyEurGrossPremium < RenEurPremOffer and PolicyRetNum = 1 

StatementMeta(, 186e685d-e26f-49b2-bbac-6be8e4f2b83e, 15, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [11]:
/* HR2.B1.O3 */
select count(distinct a.PolicyCode) from  stg.HR2B1_1_Base a left join stg.R0_HomePoliciesEligibleForRenewals b on a.PolicyCode = b.PolicyCode and PolicyRetNum = 1 
Where b.PolicyCode is null 

StatementMeta(, 186e685d-e26f-49b2-bbac-6be8e4f2b83e, 12, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [18]:
/* HR4.B2.1 HR4.B2

 */
SELECT  count(distinct a.TyPolicyCode)
FROM    stg.R0_HomePoliciesEligibleForRenewals  a,
        ods.EventStream                          d 
Where   a.TyPolicyCode = d.SourcePolicyReference
and     d.PolicyTypeGroup = 'Home'
and     EventDateTime between '2026-06-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     EventDescription in 
        (
                'New Business - Saved Document - Document Checklist SMS',
                'New Business - Saved Document - Document Checklist Email'
            )  

StatementMeta(, 186e685d-e26f-49b2-bbac-6be8e4f2b83e, 19, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [19]:
/* HR4.B2.2 
 */ 
select  count(distinct b.PolicyCode)
from    stg.R0_HomePoliciesEligibleForRenewals a, dlk.MyChill_NewUploadDocumentEvents  b 
Where   a.TyPolicyCode = b.PolicyCode
and     `Timestamp` > '2026-05-01 00:00:00.000' 
and     `Timestamp` < '2026-09-01 00:00:00.000' 
limit 100
;

StatementMeta(, 186e685d-e26f-49b2-bbac-6be8e4f2b83e, 20, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [20]:
/* HR4.B2.3 HR4.B2.6

 */
select  isAccepted, count(distinct b.PolicyCode)
from    stg.R0_HomePoliciesEligibleForRenewals a, dlk.MyChillWorkflow_DocumentStatus  b 
Where   a.TyPolicyCode = b.PolicyCode
and     `Timestamp` > '2026-07-01 00:00:00.000' 
and     `Timestamp` < '2026-09-01 00:00:00.000' 
/*and     isAccepted IsRejected */
Group by isAccepted
limit 100
;

StatementMeta(, 186e685d-e26f-49b2-bbac-6be8e4f2b83e, 21, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

In [22]:
/* HR4.B2.5
 -- doc escalation */
SELECT  distinct a.PolicyCode
FROM    stg.R0_HomePoliciesEligibleForRenewals     a,
        ods.EventStream                             d 
Where   a.TyPolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     EventDateTime > '2026-06-01 00:00:00.000' 
and     EventDateTime < '2026-09-01 00:00:00.000' 
and     EventDescription like '%Chase%' 
and     EventDescription like '%Final%'  

StatementMeta(, 186e685d-e26f-49b2-bbac-6be8e4f2b83e, 23, Finished, Available, Finished, False)

<Spark SQL result set with 445 rows and 1 fields>

In [23]:
/* R4.B2.O1 -- no doc escalation and then cancellation */
SELECT  count(distinct a.PolicyCode)
FROM    stg.R0_HomePoliciesEligibleForRenewals                   a,
        ods.EventStream                          d 
Where   a.TyPolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     EventDateTime > '2026-05-01 00:00:00.000' 
and     EventDescription in 
        (
            'Insurer Led Cancelation - Emailed Document - Reg canx email template',
            'Insurer Led Cancelation - Sent To Document Processing - Reg canx email template'
        )
and     a.TyPolicyCode in 
        (SELECT  distinct a.TyPolicyCode
        FROM    stg.R0_HomePoliciesEligibleForRenewals     a,
                ods.EventStream                             d 
        Where   a.TyPolicyCode = d.SourcePolicyReference
        and     d.EventSourceId = 3 
        and     EventDateTime > '2026-05-01 00:00:00.000' 
        and     EventDateTime < '2026-09-01 00:00:00.000' 
        and     EventDescription like '%Chase%' 
        and     EventDescription like '%Final%' ) 
        

StatementMeta(, 186e685d-e26f-49b2-bbac-6be8e4f2b83e, 24, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [34]:
/* HR3.B1.1
*/
select count(*) from  stg.R0_HomePoliciesEligibleForRenewals 
Where RenewalDate < TyReportingSaleDate 
limit 10 

StatementMeta(, 186e685d-e26f-49b2-bbac-6be8e4f2b83e, 35, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [26]:
/* 
    Ewan - please add this to event stream 
        
        but policy code / client code is sparsely populated 
        
        so we may have to match to email / phone with pii and then to ods.scv and the scv key is to be pulled? */

select	count(distinct a.PolicyCode)
from	dlk.EXT_XtremePushResults   a, stg.R0_HomePoliciesEligibleForRenewals b 
where	`timestamp` between '2026-06-01 00:00:00.000' and '2026-09-01 00:00:00.000' 
and     trim(a.PolicyCode) = trim(b.PolicyCode)
and		trim(campaign_name) in 
            (
                'Home - Renewals Chase - 10DTI',
                'Home - Renewals Chase - 20DTI',
                'Home - Renewals Chase - 2DTI'
            )
and     MessageType = 'EMAIL'
and     interaction_type = 'sent' 
limit 100 

StatementMeta(, 186e685d-e26f-49b2-bbac-6be8e4f2b83e, 27, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [30]:
select	MAX(SourceFileDate)
from	dlk.EXT_XtremePushResults_Policy   a 
where	`timestamp` between '2026-06-01 00:00:00.000' and '2026-09-01 00:00:00.000' 
and		campaign_name like '%Home%' 
and		campaign_name like '%Renewals%' 
and     MessageType in ( 'EMAIL', 'SMS')
and     interaction_type = 'sent' 
limit 100 

StatementMeta(, 186e685d-e26f-49b2-bbac-6be8e4f2b83e, 31, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [2]:
    select  *from dlk.cda_loanse

StatementMeta(, d8dc7ba4-37bb-440b-870d-46271665ae32, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 23 fields>

In [3]:
select count(*) from dlk.rbsdata2_diary

StatementMeta(, d8dc7ba4-37bb-440b-870d-46271665ae32, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
-- get chase emails from XP table
select campaign_name from ods.EXT_XtremePushResults where campaign_name like '%Chase%' 
and [timestamp] between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
Group by campaign_name
/* EXT_XtremePushResults_Policy is at policycode level -- the above is email level */ 
/*
Arrears - Chase 1 - 1 Day Past
Arrears - Chase 2 - 7 Days ago
Arrears - Chase 3 - 14 Days Past
Arrears - Chase 4 - 21 Days Past
Arrears - Chase 5 - 28 Days Past
Home - Post Renewal &amp; Purchase - xSell 
Home - Renewals Chase - 10 DTI
Home - Renewals Chase - 10 DTI - Ex-Chrome
Home - Renewals Chase - 20DTI
Home - Renewals Chase - 20DTI - Ex-Chrome
Home - Renewals Chase - 2DTI
Home - Renewals Chase - 2DTI - Exh-Chrome
Home - Post Renewal & Purchase - xSell
Home - Renewals Chase - 10DTI
Home - Renewals Chase - 20DTI
Home - Renewals Chase - 2DTI
Renewal Chase 16 Fri
Renewal Chase 16 Mon-Thur
Renewal Chase 21 Fri
Renewal Chase 21 Mon-Thur
SurveyPurchased Auto PIF
SurveyPurchased July23
Van - Renewals Chase - 10DTI
Van - Renewals Chase - 20DTI
Van - Renewals Chase - 2DTI
*/

In [ ]:
select * from stg.XPTempExtracts where 'Commercial'

        create or replace table tmp.lifelivepolicies as
        select  cast(a.PK_ContactID as varchar(255)) ClientCode,
                b.PK_ContactPolicyID
        from    dlk.life_tblcontacts        a,
                dlk.life_tblcontactpolicies b
        Where   a.PK_ContactID = b.FK_ContactID
        and     FK_PolicyStatusID = 1
        and     PolicyInceptionDate < '2026-08-29 00:00:00.000'
        AND     PolicyInceptionDate <> '1900-01-01 00:00:00.000'
        AND     PolicyMaturityDate >= '2026-08-28'
        AND     PolicyNumber IS NOT NULL
        AND     PolicyNumber <> ''
        AND     PolicyNumber <> 'GDPR'
        Order by 1

In [ ]:
select	top 100 * 
from	[msg].[Messages] 
where	RenderedMessage not like 'Hi, thanks for choosing us, we really appreciate it. We''ll be sending out your welcome pack shortly. Thanks, Chill Insurance.%' 
and		RenderedMessage not like 'Urgent! Your renewal is due tomorrow%'
and		RenderedMessage not like 'Urgent! Your renewal is due. %'
and		RenderedMessage not like 'Urgent! Your renewal is now due. %'
and		RenderedMessage <> 'Chill MFQ Chase'
and		RenderedMessage not like '%thank you for choosing Chill Insurance. We have issued your Cert and Disc.%'
and		RenderedMessage not like '%we have emailed important correspondence regarding your insurance policy %'
and		UTCDateAdded	between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 

Hi Lorraine, we have emailed important correspondence regarding your insurance policy to lorrainemurphy76@hotmail.com. Thanks, Chill Insurance.
Hi Ann Holohan, thank you for choosing Chill Insurance. We have issued your Cert and Disc.  You will have them within 5 days. To opt out text CHILL STOP to 50015.